# Rice Trait Network and Yield Driver Analysis Pipeline

Author: **Md Rezve**  
Target journals: **PLOS ONE / Scientific Reports**

This notebook integrates quantitative genetics, network analysis, and machine learning
to identify yield-driving traits in rice breeding populations.

## Workflow Overview

Field experiment data  
↓  
BLUP estimation (mixed models)  
↓  
Variance components + heritability  
↓  
Graphical Lasso trait network  
↓  
Trait interaction network  
↓  
Machine learning yield predictors  
↓  
MGIDI multi-trait selection index  
↓  
Publication-ready figures

## Install Required Libraries

In [ ]:
!pip install -q pandas numpy matplotlib seaborn networkx scikit-learn statsmodels shap openpyxl

## Import Libraries

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

from sklearn.preprocessing import StandardScaler
from sklearn.covariance import GraphicalLassoCV
from sklearn.linear_model import LinearRegression
from sklearn.cross_decomposition import PLSRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import r2_score
from sklearn.decomposition import FactorAnalysis

import statsmodels.formula.api as smf
import shap

warnings.filterwarnings('ignore')

## Upload Dataset

In [ ]:
from google.colab import files
uploaded = files.upload()

filename = list(uploaded.keys())[0]

if filename.endswith('.xlsx'):
    df = pd.read_excel(filename)
else:
    df = pd.read_csv(filename)

df.head()

## Identify Traits

In [ ]:
GENOTYPE_COL = 'Genotype'
REPLICATION_COL = 'Replication'

traits = [c for c in df.columns if c not in [GENOTYPE_COL, REPLICATION_COL]]

genotypes = df[GENOTYPE_COL].unique()
replications = df[REPLICATION_COL].unique()

print('Traits:', traits)

## BLUP Estimation (Mixed Model)

In [ ]:
blup = pd.DataFrame(index=genotypes, columns=traits)

for trait in traits:

    formula = f'Q("{trait}") ~ C({REPLICATION_COL})'

    md = smf.mixedlm(formula, df, groups=df[GENOTYPE_COL])

    m = md.fit()

    mu = m.fe_params[0]

    for g in genotypes:

        u = float(np.asarray(m.random_effects.get(g, [0]))[0])

        blup.loc[g, trait] = mu + u

blup.head()

## Standardize BLUP Matrix

In [ ]:
Z = pd.DataFrame(
    StandardScaler().fit_transform(blup),
    columns=blup.columns,
    index=blup.index
)

## Partial Correlation Network (Graphical Lasso)

In [ ]:
gl = GraphicalLassoCV()

gl.fit(Z.values)

P = gl.precision_

d = np.sqrt(np.diag(P))

pcorr = -P / np.outer(d, d)

np.fill_diagonal(pcorr, 1)

pcorr_df = pd.DataFrame(pcorr, index=traits, columns=traits)

## Partial Correlation Heatmap

In [ ]:
plt.figure(figsize=(10,8))
sns.heatmap(pcorr_df, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Partial Correlation Heatmap')
plt.show()

## Trait Interaction Network

In [ ]:
G = nx.Graph()

for i in range(len(traits)):
    for j in range(i+1,len(traits)):

        r = pcorr_df.iloc[i,j]

        if abs(r) >= 0.2:
            G.add_edge(traits[i], traits[j], weight=r)

pos = nx.spring_layout(G, seed=42)

plt.figure(figsize=(10,8))

nx.draw(G, pos, with_labels=True, node_size=2000)

plt.title('Trait Interaction Network')

plt.show()

## Multiple Linear Regression — Yield Drivers

In [ ]:
yield_trait = 'Grain yield per hill'

X = Z.drop(columns=[yield_trait])

y = Z[yield_trait]

lin = LinearRegression().fit(X,y)

importance = pd.Series(np.abs(lin.coef_), index=X.columns)

importance.sort_values(ascending=False)

## Random Forest Feature Importance

In [ ]:
rf = RandomForestRegressor(n_estimators=300, random_state=42)

rf.fit(X,y)

rf_importance = pd.Series(rf.feature_importances_, index=X.columns)

rf_importance.sort_values(ascending=False)

## SHAP Explainable AI

In [ ]:
gb = GradientBoostingRegressor()

gb.fit(X,y)

explainer = shap.Explainer(gb,X)

shap_values = explainer(X)

shap.plots.bar(shap_values)

## MGIDI Multi-Trait Selection Index

In [ ]:
gmeans = df.groupby(GENOTYPE_COL)[traits].mean()

Zg = StandardScaler().fit_transform(gmeans)

fa = FactorAnalysis(n_components=min(5,len(traits)))

F = fa.fit_transform(Zg)

ideotype = F.max(axis=0)

mgidi = np.sqrt(((F - ideotype)**2).sum(axis=1))

mgidi = pd.Series(mgidi, index=gmeans.index).sort_values()

mgidi.head()